# Customer Purchase Funnel: Conditional Probability Analysis

This notebook analyzes how customer conversion probability changes
as users move through different stages of a purchase funnel.

The focus is on understanding conditional probability in a real business context.

In [9]:
import pandas as pd

df = pd.read_csv("../data/customer_journey.csv")
df.head()


,SessionID,UserID,Timestamp,PageType,DeviceType,Country,ReferralSource,TimeOnPage_seconds,ItemsInCart,Purchased
0,session_0,user_2223,2025-01-20 22:53:34,home,Desktop,India,Social Media,55,0,0
1,session_1,user_2192,2025-02-26 12:57:10,home,Tablet,Germany,Email,99,0,0
2,session_1,user_2192,2025-02-26 12:59:11,product_page,Tablet,Germany,Email,121,0,0
3,session_2,user_1708,2025-06-24 15:40:46,home,Mobile,India,Google,160,0,0
4,session_3,user_2976,2025-06-11 07:21:02,home,Tablet,UK,Google,113,0,0


### Q1. Overall Purchase Probability

What is the overall probability that a session results in a purchase?

This serves as the baseline conversion rate against which all conditional
probabilities will be compared.

In [10]:
# Overall probability of purchase (baseline)
overall_purchase_probability = df["Purchased"].mean()
overall_purchase_probability

np.float64(0.39704379275100243)

### Interpretation

The overall probability of purchase is approximately **39.7%**.

This means that, across all recorded sessions, roughly **4 out of 10 sessions**
ultimately result in a purchase.

This value acts as a **baseline conversion rate**. On its own, it does not explain
*why* users convert, but it provides essential context for evaluating how different
conditions (such as page stage, cart activity, or engagement) increase or decrease
the likelihood of purchase.

### Q2. Purchase Probability by Page Stage

What is the probability that a session results in a purchase,
**given that the session reached a particular page stage**?

This helps understand how purchase likelihood changes
as users move through the funnel.

In [14]:
# Purchase probability conditioned on page stage
purchase_probability_by_stage = (
    df.groupby("PageType")["Purchased"]
      .mean()
      .sort_values(ascending=False)
)

purchase_probability_by_stage

PageType
confirmation    1.000000
checkout        0.899377
cart            0.631645
product_page    0.253323
home            0.202000
Name: Purchased, dtype: float64

### Interpretation

Purchase probability increases steadily as sessions move deeper into the funnel.

- Sessions that reached the **home** page have a purchase probability of about **20.2%**.
- This increases slightly to **25.3%** for sessions that reached a **product page**.
- Once a session reaches the **cart**, purchase probability rises sharply to **63.2%**.
- At the **checkout** stage, nearly **90%** of sessions result in a purchase.
- Sessions that reached the **confirmation** page have a purchase probability of **100%**, as expected.

This progression confirms that page stage is a **strong signal of purchase intent**.
Early stages represent browsing behavior, while later stages represent commitment.

The sharp increase between **product page → cart** highlights a critical transition
where user intent becomes significantly stronger.

### Q3. Purchase Probability Based on Items in Cart

How does the probability of purchase change based on the number of items
present in the cart?

This question evaluates whether cart size acts as a meaningful signal
of purchase intent.


In [16]:
# Create cart size buckets
df["CartBucket"] = pd.cut(
    df["ItemsInCart"],
    bins=[-1, 0, 1, 3, df["ItemsInCart"].max()],
    labels=["0 items", "1 item", "2–3 items", "4+ items"]
)

# Purchase probability by cart bucket
purchase_probability_by_cart = (
    df.groupby("CartBucket")["Purchased"]
      .mean()
)

purchase_probability_by_cart

C:\Users\dell\AppData\Local\Temp\ipykernel_17268\2257530111.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("CartBucket")["Purchased"]


CartBucket
0 items      0.325828
1 item       0.533188
2–3 items    0.520731
4+ items     0.497323
Name: Purchased, dtype: float64

### Interpretation

Purchase probability increases significantly once a customer has at least one item in the cart.

- Sessions with **0 items in cart** have a purchase probability of about **32.6%**, which is below the overall baseline.
- When there is **1 item in the cart**, purchase probability jumps sharply to **53.3%**.
- Sessions with **2–3 items** show a similar purchase probability (**~52.1%**).
- Sessions with **4 or more items** do not show further improvement, with purchase probability slightly lower at **~49.7%**.

This suggests that **the presence of items in the cart matters more than the exact number of items**.
Adding the first item represents a strong intent signal, while additional items do not proportionally increase the likelihood of purchase.

### Q4. Purchase Probability Given Page Stage and Cart Activity

How does purchase probability change when **page stage** and **cart activity**
are considered together?

This question evaluates whether combining funnel stage with cart information
provides a stronger signal of purchase intent than either factor alone.


In [17]:
# Focus on sessions with at least one item in cart
df_with_cart = df[df["ItemsInCart"] > 0]

# Purchase probability by page stage, given items in cart
purchase_prob_stage_and_cart = (
    df_with_cart.groupby("PageType")["Purchased"]
                .mean()
                .sort_values(ascending=False)
)

purchase_prob_stage_and_cart

PageType
checkout        0.899377
cart            0.634802
product_page    0.261165
Name: Purchased, dtype: float64

### Interpretation

When restricting analysis to sessions that already have **at least one item in the cart**,
page stage continues to play a strong role in purchase likelihood.

- Sessions at the **product page** stage with items in the cart have a purchase probability of about **26.1%**.
- Once a session reaches the **cart** page with items present, purchase probability increases sharply to **63.5%**.
- Sessions that reach the **checkout** stage with items in the cart convert at nearly **90%**, indicating very strong purchase intent.

This shows that **cart presence alone is not sufficient** to explain purchase behavior.
Instead, **cart activity combined with funnel progression** provides a much stronger and more reliable signal of purchase intent.

### Q5. Purchase Probability Based on Time Spent on Page

Does higher engagement, measured as **time spent on a page**, increase the
probability that a session results in a purchase?

To answer this, sessions are grouped into **low** and **high** engagement
based on time spent on page.

In [21]:
import numpy as np
# Calculate median time on page
median_time = df["TimeOnPage_seconds"].median()

# Create engagement buckets
df["TimeBucket"] = np.where(
    df["TimeOnPage_seconds"] > median_time,
    "High Time on Page",
    "Low Time on Page"
)

# Purchase probability by time bucket
purchase_probability_by_time = (
    df.groupby("TimeBucket")["Purchased"]
      .mean()
)

purchase_probability_by_time

TimeBucket
High Time on Page    0.394958
Low Time on Page     0.399095
Name: Purchased, dtype: float64

### Interpretation

Time spent on a page does **not** appear to be a strong standalone indicator of purchase.

- Sessions with **high time on page** have a purchase probability of approximately **39.5%**.
- Sessions with **low time on page** have a very similar purchase probability of about **39.9%**.

The near-identical probabilities suggest that **time on page alone does not meaningfully distinguish purchasing behavior** in this dataset.
Higher engagement, as measured purely by time spent, does not necessarily translate into higher purchase intent.

### Q.6 Final Question. Which Factor Provides the Strongest Signal of Purchase?

Among page stage, cart activity, and time spent on page,
which single factor most strongly increases the probability of purchase?

This question synthesizes all previous analysis to identify
the most informative signal of customer intent.


In [22]:
baseline = overall_purchase_probability

signal_strength = {
    "Best Page Stage (Checkout)": purchase_probability_by_stage.loc["checkout"] / baseline,
    "Items in Cart (>0)": purchase_probability_by_cart.loc["1 item"] / baseline,
    "High Time on Page": purchase_probability_by_time.loc["High Time on Page"] / baseline
}

signal_strength

{'Best Page Stage (Checkout)': np.float64(2.2651825467497773),
 'Items in Cart (>0)': np.float64(1.3428953123821632),
 'High Time on Page': np.float64(0.9947466511357018)}

### Final Interpretation: Strongest Signal of Purchase

Among the three factors analyzed: page stage, cart activity, and time spent on page 
**(page stage provides the strongest signal of purchase intent)**.

- Reaching the **checkout stage** increases purchase probability by approximately **2.27×**
  compared to the overall baseline.
- Having **items in the cart** increases purchase probability by about **1.34×**,
  indicating a meaningful but weaker signal.
- **High time on page** provides virtually **no lift over baseline** (≈1.0×),
  suggesting it is not a reliable standalone indicator of purchase intent.

This comparison shows that **progress through the funnel** is the most informative
single factor for predicting purchase, followed by cart activity.
Engagement metrics such as time spent on page are comparatively weak and potentially misleading
when used in isolation.